### Imports

In [ ]:
# Import libraries for data splitting
import splitfolders

# Import libraries for computer vision and deep learning
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models
import torch.nn as nn
import torch

# Import libraries for progress bars and metrics
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Import PIL to handle potentially truncated images
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

### Split to train, val, test

In [ ]:
# splitfolders.ratio("data", output="split_data", 
#                    seed=42, ratio=(.7, .15, .15), group_prefix=None, move=False)

Copying files: 8022 files [02:40, 50.05 files/s]


### Transforms & Loading Setup

In [ ]:
# Define data augmentation and preprocessing transformations
# Training transforms include augmentation for better generalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224 for ResNet input
    transforms.RandomHorizontalFlip(),  # Randomly flip images horizontally
    transforms.RandomRotation(10),  # Randomly rotate images by up to 10 degrees
    transforms.ToTensor(),  # Convert images to PyTorch tensors
    transforms.Normalize(  # Normalize using ImageNet mean and std
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])

# Validation and test transforms without augmentation
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224
    transforms.ToTensor(),  # Convert to tensors
    transforms.Normalize(  # Normalize with same values
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)
])

In [3]:
# Load the dataset
train_data = datasets.ImageFolder(root='split_data/train', transform=train_transform)
val_data = datasets.ImageFolder(root='split_data/val', transform=val_test_transform)
test_data = datasets.ImageFolder(root='split_data/test', transform=val_test_transform)

In [4]:
# Create a DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [5]:
# Accessing metadata
print(f"Classes found: {train_data.classes}") 
print(f"Class to index mapping: {train_data.class_to_idx}")

Classes found: ['asteroid', 'black hole', 'comet', 'galaxy', 'nebula', 'planet', 'star']
Class to index mapping: {'asteroid': 0, 'black hole': 1, 'comet': 2, 'galaxy': 3, 'nebula': 4, 'planet': 5, 'star': 6}


### Setup and Train The Model

In [ ]:
# Load pre-trained ResNet18 model with ImageNet weights
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Get the number of input features to the final fully connected layer
num_features = model.fc.in_features  

# Replace the final layer with a new one for 7 classes (our astronomy categories)
model.fc = nn.Linear(num_features, 7)

In [7]:
# Define the loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# Training loop with validation and best model saving
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    """
    Train the model for a specified number of epochs with validation.
    
    Args:
        model: The PyTorch model to train
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        criterion: Loss function
        optimizer: Optimizer for updating model parameters
        num_epochs: Number of training epochs
        device: Device to run training on (CPU or GPU)
    
    Returns:
        None (prints training progress and saves best model)
    """
    best_val_acc = 0.0
    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader with progress bar
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False):
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader with progress bar
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False):
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)
        
        # Save the model if validation accuracy improves
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

        # Print the results for the current epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [9]:
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [15]:
train(model, train_loader, val_loader, criterion, optimizer, num_epochs=10,device=device)

Epoch [1/10], train loss: 0.6867, train acc: 0.7625, val loss: 0.5414, val acc: 0.8167


Epoch [2/10], train loss: 0.4869, train acc: 0.8284, val loss: 0.4625, val acc: 0.8383


Epoch [3/10], train loss: 0.4102, train acc: 0.8520, val loss: 0.4221, val acc: 0.8525


Epoch [4/10], train loss: 0.3431, train acc: 0.8762, val loss: 0.4023, val acc: 0.8550


Epoch [5/10], train loss: 0.2796, train acc: 0.8976, val loss: 0.3556, val acc: 0.8750


Epoch [6/10], train loss: 0.2585, train acc: 0.9082, val loss: 0.3783, val acc: 0.8567


Epoch [7/10], train loss: 0.2046, train acc: 0.9284, val loss: 0.3660, val acc: 0.8692


Epoch [8/10], train loss: 0.1762, train acc: 0.9371, val loss: 0.3402, val acc: 0.8825


Epoch [9/10], train loss: 0.1675, train acc: 0.9400, val loss: 0.3551, val acc: 0.8767


Epoch [10/10], train loss: 0.1425, train acc: 0.9507, val loss: 0.3528, val acc: 0.8792


### Evaluate the Model

In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Evaluate the trained model on the test set and compute metrics.
    
    Args:
        model: The trained PyTorch model
        test_loader: DataLoader for test data
        device: Device to run evaluation on (CPU or GPU)
    
    Returns:
        None (prints evaluation metrics)
    """
    # Initialize dictionaries to store correct and total predictions
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    # Set the model to evaluation mode
    model.eval()

    # Track the ground truth labels and predictions
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", leave=False):
            # Move the inputs and labels to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Collect predictions and labels for metric calculations
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # Update the correct and total predictions
            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    # Calculate accuracy per class
    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname] if total_pred[classname] > 0 else 0
                          for classname in test_loader.dataset.classes}

    # Calculate precision, recall, and F1 per class
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=list(range(len(test_loader.dataset.classes))),
        zero_division=0
    )

    # Calculate overall accuracy
    overall_accuracy = accuracy_score(all_labels, all_preds)

    # Print the evaluation results
    print("Metrics per class:")
    for idx, classname in enumerate(test_loader.dataset.classes):
        print(f"{classname}: accuracy={accuracy_per_class[classname]:.4f}, precision={precision[idx]:.4f}, recall={recall[idx]:.4f}, f1={f1[idx]:.4f}")

    print()
    print(f"Overall Accuracy: {overall_accuracy:.4f}")

In [26]:
evaluate_model(model, test_loader, device)

Metrics per class:
asteroid: accuracy=0.8333, precision=0.8750, recall=0.8333, f1=0.8537
black hole: accuracy=0.8182, precision=0.8351, recall=0.8182, f1=0.8265
comet: accuracy=0.8730, precision=0.8594, recall=0.8730, f1=0.8661
galaxy: accuracy=0.9515, precision=0.9120, recall=0.9515, f1=0.9313
nebula: accuracy=0.8667, precision=0.9070, recall=0.8667, f1=0.8864
planet: accuracy=0.9099, precision=0.9619, recall=0.9099, f1=0.9352
star: accuracy=0.8341, precision=0.8111, recall=0.8341, f1=0.8224

Overall Accuracy: 0.8916
